In [1]:
data_root = "/home/jackson-devworks/Desktop/raining_prediction/training_data"

# 1. Prepare dataset

In [2]:
import pandas as pd

In [3]:
csv_file_path = "/home/jackson-devworks/Desktop/raining_prediction/training_data/LV_BanVe.csv"
csv_ouput_path = "new_data_with_predictions_BanVe.csv"
df_property = ["COMS","GFS","MITSUISHI2011_D03","LINKF_D03","LINBMJ_D03","ETAKF_D03","ETAG3_D03","ETABMJ_D03","Rainfall"]
df = pd.read_csv(csv_file_path)

df["Session"] = pd.to_datetime(df["Session"])
df["Datetime"] = pd.to_datetime(df["Datetime"])
df["Time_difference"] = (df["Datetime"] - df["Session"]).dt.total_seconds() / 3600
df = df.drop(columns=['Session', 'Datetime'])
df = df[df_property]
df = df.dropna()


In [4]:
import numpy as np

# Assuming df is a pandas DataFrame
X = df.iloc[:, :-1]  # Select all rows and all columns except the last one
y = df.iloc[:, -1]   # Select all rows and the last column (change 1 to -1 for consistency)
X = np.array(X)
y = np.array(y)


In [5]:
print(X.shape)
print(y.shape)

(54264, 8)
(54264,)


# 2. Machine learning

In [6]:
from sklearn.model_selection import train_test_split
from lazypredict.Supervised import LazyRegressor


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Kiểm tra kích thước
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (43411, 8)
y_train shape: (43411,)
X_test shape: (10853, 8)
y_test shape: (10853,)


In [8]:
# lazy_model = LazyRegressor()

# # Áp dụng LazyRegressor để thử nhiều mô hình
# models, predictions = lazy_model.fit(X_train, X_valid, y_train, y_valid)

# # Hiển thị kết quả (các mô hình và hiệu suất của chúng)
# print(models)

# 3. Ensemble Learning

## a. RandomForestRegressor

In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [10]:
# Định nghĩa mô hình MLPRegressor
random_forest_regressor = RandomForestRegressor(random_state=42)

# Định nghĩa các tham số để tìm kiếm
param_grid = { 
            "n_estimators"      : [10,20,30],
            "max_features"      : ["auto", "sqrt", "log2"],
            "min_samples_split" : [2,4,8],
            "bootstrap": [True, False],
            }

# Khởi tạo GridSearchCV
grid_search = GridSearchCV(estimator=random_forest_regressor, param_grid=param_grid, cv=5, n_jobs=-1, verbose=0)

# Huấn luyện mô hình với GridSearchCV
grid_search.fit(X_train, y_train)

# In ra các tham số tối ưu
print(f"Best parameters: {grid_search.best_params_}")

# Dự đoán trên tập test với mô hình tốt nhất
best_random_forest_regressor = grid_search.best_estimator_
y_pred = best_random_forest_regressor.predict(X_test)

# Đánh giá mô hình
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

Best parameters: {'bootstrap': True, 'max_features': 'sqrt', 'min_samples_split': 8, 'n_estimators': 30}
MAE: 0.5640507942186187
MSE: 0.9943247288328428
R2: 0.08391425457381563


## b. BaggingRegressor

In [11]:
from sklearn.ensemble import BaggingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [12]:
# Step 1: Define the model
bagging_regressor = BaggingRegressor()  

# Step 2: Set the parameter grid, including `n_nonzero_coefs` as a possible hyperparameter
param_grid = {
    'n_estimators': [10,20,30]
}
# Step 3: Set up GridSearchCV
grid_search = GridSearchCV(estimator=bagging_regressor, param_grid=param_grid, cv=5, n_jobs=-1, verbose=1)

# Step 4: Fit the model with the grid search
grid_search.fit(X_train, y_train)

# Step 5: Print the best parameters and best model
print(f"Best parameters: {grid_search.best_params_}")

# Evaluate the best model
best_bagging_regressor = grid_search.best_estimator_
y_pred = best_bagging_regressor.predict(X_test)

# Step 6: Evaluate the model
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

Fitting 5 folds for each of 3 candidates, totalling 15 fits
Best parameters: {'n_estimators': 30}
MAE: 0.5791251317971172
MSE: 1.0286550063697741
R2: 0.05228527364218016


## c.XGBRegressor

In [13]:
from xgboost.sklearn import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [14]:
xgbregressor = XGBRegressor()

# Định nghĩa các tham số cần tìm kiếm
param_grid = {'nthread':[4], #when use hyperthread, xgboost may become slower
              'learning_rate': [.03, 0.05, .07], #so called `eta` value
              'max_depth': [5, 6, 7],
              'min_child_weight': [4],
              'subsample': [0.7],
              'colsample_bytree': [0.7],
              'n_estimators': [500]
              }

# Khởi tạo GridSearchCV
grid_search = GridSearchCV(estimator=xgbregressor, param_grid=param_grid, cv=5, n_jobs=-1, verbose=0)

# Huấn luyện mô hình với GridSearchCV
grid_search.fit(X_train, y_train)

# In ra các tham số tối ưu
print(f"Best parameters: {grid_search.best_params_}")

# Dự đoán trên tập test với mô hình tốt nhất
best_xgbregressor = grid_search.best_estimator_
y_pred = best_xgbregressor.predict(X_test)

# Đánh giá mô hình
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

Best parameters: {'colsample_bytree': 0.7, 'learning_rate': 0.03, 'max_depth': 6, 'min_child_weight': 4, 'n_estimators': 500, 'nthread': 4, 'subsample': 0.7}
MAE: 0.5483028707634272
MSE: 0.9885145299439093
R2: 0.08926727479537921


## d. LGBMRegressor

In [15]:
from sklearn.model_selection import GridSearchCV
import lightgbm as lgb
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [16]:
# Khởi tạo mô hình LassoLarsIC với tiêu chí AIC hoặc BIC
lgbmregresssor = lgb.LGBMRegressor()  # Bạn có thể chọn 'bic' nếu muốn

param_grid = {
    'task' : ['predict'],
    'boosting': ['gbdt' ],
    'objective': ['root_mean_squared_error'], 
}

# Huấn luyện mô hình
grid_search = GridSearchCV(estimator=lgbmregresssor, param_grid=param_grid, cv=5, n_jobs=-1, verbose=0)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")

# Dự đoán trên tập test
best_lgbmregresssor = grid_search.best_estimator_
y_pred = best_lgbmregresssor.predict(X_test)

# In ra các tham số và hệ số mô hình đã được tối ưu


# Đánh giá mô hình
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

[LightGBM] [Warning] boosting is set=gbdt, boosting_type=gbdt will be ignored. Current value: boosting=gbdt
[LightGBM] [Warning] boosting is set=gbdt, boosting_type=gbdt will be ignored. Current value: boosting=gbdt
[LightGBM] [Warning] boosting is set=gbdt, boosting_type=gbdt will be ignored. Current value: boosting=gbdt
[LightGBM] [Warning] boosting is set=gbdt, boosting_type=gbdt will be ignored. Current value: boosting=gbdt
[LightGBM] [Warning] boosting is set=gbdt, boosting_type=gbdt will be ignored. Current value: boosting=gbdt
[LightGBM] [Warning] boosting is set=gbdt, boosting_type=gbdt will be ignored. Current value: boosting=gbdt
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001912 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2040
[LightGBM] [Info] Number of data points in the train set: 34728, number of used features: 8
[LightGBM] [Info] Start training from score 0.464493
[LightGBM] 

## e. HistGradientBoostingRegressor

In [17]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [18]:
# Khởi tạo mô hình LarsCV với số lượng fold cross-validation (ví dụ: 5)
hist_gradient_boosting_regressor = HistGradientBoostingRegressor()

param_grid = {
    'learning_rate': [0.01, 0.1, 1],
    'max_iter': [50, 100],
    'max_leaf_nodes': [10, 20, 30]
}
grid_search = GridSearchCV(estimator=hist_gradient_boosting_regressor, param_grid=param_grid, cv=5, n_jobs=-1, verbose=0)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")

# Dự đoán trên tập test
best_hist_gradient_boosting_regressor = grid_search.best_estimator_
y_pred = best_hist_gradient_boosting_regressor.predict(X_test)

# Đánh giá mô hình
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

Best parameters: {'learning_rate': 0.1, 'max_iter': 100, 'max_leaf_nodes': 30}
MAE: 0.5515006276477749
MSE: 0.9886692780338331
R2: 0.08912470314328169


## f. Ensemble

In [19]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression

In [20]:
base_models = [
    ('RandomForestRegressor', RandomForestRegressor(
        random_state=42,bootstrap=False, max_features='sqrt', 
        min_samples_split=2, n_estimators=30
        )
    ), 
    ('BaggingRegressor', BaggingRegressor(
                        n_estimators=20
                    )
    ),
    ('XGBRegressor', XGBRegressor(
        colsample_bytree=0.7, learning_rate=0.07, max_depth=7, min_child_weight=4,
        n_estimators = 500, nthread=4, subsample=0.7
        )
    ),
    ('LGBMRegressor', lgb.LGBMRegressor(
        boosting='gbdt', objective='root_mean_squared_error', task='predict',
        force_col_wise=True
        )
    ),
    ('HistGradientBoostingRegressor', HistGradientBoostingRegressor(
        learning_rate=0.1, max_iter=100, max_leaf_nodes=30
        )
    )
]

# Định nghĩa mô hình stacking với Linear Regression là meta-model
stacking_model = StackingRegressor(estimators=base_models, final_estimator=LinearRegression())

# Huấn luyện mô hình stacking
stacking_model.fit(X_train, y_train)

# Dự đoán và đánh giá mô hình
y_pred = stacking_model.predict(X_test)

# Đánh giá kết quả
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

[LightGBM] [Warning] boosting is set=gbdt, boosting_type=gbdt will be ignored. Current value: boosting=gbdt
[LightGBM] [Warning] boosting is set=gbdt, boosting_type=gbdt will be ignored. Current value: boosting=gbdt
[LightGBM] [Info] Total Bins 2040
[LightGBM] [Info] Number of data points in the train set: 43411, number of used features: 8
[LightGBM] [Info] Start training from score 0.464647
[LightGBM] [Warning] boosting is set=gbdt, boosting_type=gbdt will be ignored. Current value: boosting=gbdt
[LightGBM] [Warning] boosting is set=gbdt, boosting_type=gbdt will be ignored. Current value: boosting=gbdt
[LightGBM] [Info] Total Bins 2040
[LightGBM] [Info] Number of data points in the train set: 34728, number of used features: 8
[LightGBM] [Info] Start training from score 0.464493
[LightGBM] [Warning] boosting is set=gbdt, boosting_type=gbdt will be ignored. Current value: boosting=gbdt
[LightGBM] [Warning] boosting is set=gbdt, boosting_type=gbdt will be ignored. Current value: boosting

In [21]:
import pandas as pd
import numpy as np

# Generate predictions from all models
ensemble_pred = stacking_model.predict(X_test)
rf_pred = best_random_forest_regressor.predict(X_test)
bagging_pred = best_bagging_regressor.predict(X_test)
xgb_pred = best_xgbregressor.predict(X_test)
lgbm_pred = best_lgbmregresssor.predict(X_test)  # Fix the typo
hist_pred = best_hist_gradient_boosting_regressor.predict(X_test)

# Collect predictions into a DataFrame
predictions_df = pd.DataFrame({
    "Rainfall": y_test,  # Assuming y_test is an array or pandas Series
    "Ensemble_predictions": ensemble_pred,
    "RandomForestRegressor_predictions": rf_pred,
    "BaggingRegressor_pred": bagging_pred,
    "XGBRegressor_pred": xgb_pred,
    "LGBMRegressor_pred": lgbm_pred,
    "HistGradientBoostingRegressor_pred": hist_pred
})

ref_df = df[df_property[:-1]]

total_df = pd.concat([predictions_df, ref_df], axis=1)
# Save to file
total_df.to_csv(csv_ouput_path, index=False)

# Optionally, display the first few rows
print(predictions_df.head())


[LightGBM] [Warning] boosting is set=gbdt, boosting_type=gbdt will be ignored. Current value: boosting=gbdt
[LightGBM] [Warning] boosting is set=gbdt, boosting_type=gbdt will be ignored. Current value: boosting=gbdt
   Rainfall  Ensemble_predictions  RandomForestRegressor_predictions  \
0      0.00                  0.33                               0.28   
1      0.00                  0.33                               0.26   
2      0.00                  0.23                               0.18   
3      1.30                  1.22                               1.41   
4      0.70                  0.17                               0.08   

   BaggingRegressor_pred  XGBRegressor_pred  LGBMRegressor_pred  \
0                   0.36               0.29                0.29   
1                   0.23               0.29                0.32   
2                   0.21               0.24                0.18   
3                   1.29               1.42                1.10   
4               